# E2 — Popularity Bias Analysis (SNA-MXH)

**Input:** Amazon-book `trn_mat.pkl`, `tst_mat.pkl` và 7 file `topk_test.npz` đã xuất ở E1.  
**Output:** bảng per-run, mean ± SD theo model, kiểm tra nhất quán với E1, biểu đồ báo cáo và tài liệu protocol.  
**Không training / không inference / không thay đổi checkpoint hay Top-K cũ.**

### Evaluation protocol đã chốt

| Thành phần | Quy ước E2 |
|---|---|
| Baselines | Most Popular (1 run), LightGCN (3 seeds), RLMRec-Con (`lightgcn_plus`, 3 seeds) |
| Dataset / split | Amazon-book; dùng đúng Train/Test từ E1 |
| Recommendation cutoff | K = 10, 20 (cắt từ Top-20 theo đúng thứ hạng) |
| Ranking candidate pool | **Train-only masking**: đã loại item user tương tác ở Train trong E1; Validation không mask thêm |
| Item popularity | `degree(i) = số user tương tác item i trong Train`, đếm tương tác nhị phân |
| Head / Tail | Sắp xếp giảm dần theo train degree, hòa điểm theo item ID tăng dần; **Head = ceil(20% × n_items)**, Tail = phần còn lại |
| ARP@K | Trung bình (train-degree của K đề xuất) theo từng user rồi trung bình giữa users |
| APLT@K | Trung bình tỷ lệ Tail trong K đề xuất theo từng user rồi trung bình giữa users |
| Head Recall@K / Tail Recall@K | **Macro Recall có điều kiện**: chỉ trung bình trên test users có ít nhất 1 test ground-truth Head/Tail item; báo cáo `n_eligible_users` |
| Degree–Exposure | Exposure của item = số lần được gợi ý trong Top-K của test users; tương quan Spearman giữa train degree và exposure trên **toàn catalog** |
| Tổng hợp qua seeds | `mean ± SD`, population SD (`ddof=0`), thống nhất E1; Most Popular chỉ báo một giá trị (không có SD) |

**Diễn giải:** ARP thấp, APLT cao chưa đủ chứng minh hệ gợi ý tốt hơn. Luôn xem chung Recall/NDCG từ E1, đặc biệt Tail Recall ở E2.

**Reference implementation:** [HKUDS/RLMRec — data handler](https://github.com/HKUDS/RLMRec/blob/main/encoder/data_utils/data_handler_general_cf.py). Việc đánh giá Test sử dụng `AllRankTstData(tst_mat, trn_mat)` trong code công khai. Notebook này chỉ *đọc* các ranking Top-K đã xuất theo protocol đó.

## CELL 1 — Libraries, Google Drive và đường dẫn
Chạy trên Google Colab. Nếu chạy local/Jupyter, đổi `ROOT` thành thư mục dự án tương ứng. Folder E2 được tạo tự động.

In [ ]:
import json
import math
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.stats import spearmanr
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running on Google Colab; using local ROOT if configured.')

ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
DATA_DIR = ROOT / 'data/raw/amazon'
PRED_DIR = ROOT / 'experiments/predictions'
E1_SUMMARY_PATH = ROOT / 'experiments/E1/summary/e1_baseline_summary.json'
OUT_DIR = ROOT / 'experiments/E2/popularity_bias'
OUT_DIR.mkdir(parents=True, exist_ok=True)

K_VALUES = (10, 20)
HEAD_FRACTION = 0.20
EXPECTED_EXPORTS = {
    'Most Popular': {'-': PRED_DIR / 'most_popular/topk_test.npz'},
    'LightGCN': {str(s): PRED_DIR / f'lightgcn/seed_{s}/topk_test.npz' for s in (2023, 2024, 2025)},
    'RLMRec-Con': {str(s): PRED_DIR / f'lightgcn_plus/seed_{s}/topk_test.npz' for s in (2023, 2024, 2025)},
}
MODELS = list(EXPECTED_EXPORTS)

for file in [DATA_DIR/'trn_mat.pkl', DATA_DIR/'tst_mat.pkl']:
    assert file.is_file(), f'Missing dataset: {file}'
for model, runs in EXPECTED_EXPORTS.items():
    for seed, path in runs.items():
        assert path.is_file(), f'Missing Top-K export for {model} seed {seed}: {path}'
print('ROOT:', ROOT)
print('Number of Top-K exports:', sum(map(len, EXPECTED_EXPORTS.values())))
print('Output folder:', OUT_DIR)

## CELL 2 — Load đúng interaction matrices của RLMRec
Load pickle dạng sparse/dense; chuyển về CSR **nhị phân**, giữ nguyên indexing của RLMRec. `Train` để xác định popularity và masking; `Test` để xác định ground truth.

In [ ]:
def load_binary_csr(path):
    with open(path, 'rb') as f:
        mat = pickle.load(f)
    if sp.issparse(mat):
        csr = mat.tocsr(copy=True)
        csr.sum_duplicates()
        csr.data = (csr.data != 0).astype(np.int8)
        csr.eliminate_zeros()
    else:
        arr = np.asarray(mat)
        assert arr.ndim == 2, f'Expected 2-D interactions in {path}'
        csr = sp.csr_matrix((arr != 0).astype(np.int8))
    csr.sort_indices()
    return csr

train = load_binary_csr(DATA_DIR/'trn_mat.pkl')
test = load_binary_csr(DATA_DIR/'tst_mat.pkl')
assert train.shape == test.shape, f'Split shapes mismatch: {train.shape} != {test.shape}'
n_users, n_items = train.shape
active_test_users = np.flatnonzero(np.diff(test.indptr) > 0)
assert len(active_test_users) > 0

overlap = train.multiply(test).nnz
if overlap:
    print(f'WARNING: {overlap} Train/Test user-item overlaps found. Inspect split protocol.')

print(f'Users: {n_users:,}; Items: {n_items:,}')
print(f'Train binary interactions: {train.nnz:,}')
print(f'Test binary interactions: {test.nnz:,}')
print(f'Test users with ground truth: {len(active_test_users):,}')
print(f'Train/Test overlaps: {overlap:,}')

## CELL 3 — Train popularity & Head/Tail
Đếm **user-item interactions từ Train duy nhất**. `ceil(0.2 * 9332) = 1867` Head nếu catalog còn đủ 9.332 items. Tie-break bằng ID để tái lập được.

In [ ]:
item_degree = np.asarray(train.sum(axis=0)).ravel().astype(np.int64)
assert len(item_degree) == n_items
n_head = math.ceil(HEAD_FRACTION * n_items)
item_order = np.lexsort((np.arange(n_items), -item_degree))
head_mask = np.zeros(n_items, dtype=bool)
head_mask[item_order[:n_head]] = True
tail_mask = ~head_mask

item_popularity = pd.DataFrame({
    'item_id': np.arange(n_items),
    'train_degree': item_degree,
    'group': np.where(head_mask, 'Head', 'Tail'),
})
item_popularity.to_csv(OUT_DIR/'E2_item_popularity.csv', index=False)

print(f'Head: {int(head_mask.sum()):,} items ({head_mask.mean():.2%})')
print(f'Tail: {int(tail_mask.sum()):,} items ({tail_mask.mean():.2%})')
print('Catalog items with zero train degree:', int((item_degree == 0).sum()))
print('Share of Train interactions on Head items:',
      f'{item_degree[head_mask].sum() / max(1, item_degree.sum()):.2%}')
display(item_popularity.sort_values(['train_degree','item_id'], ascending=[False,True]).head(10))

## CELL 4 — Đọc & audit toàn bộ 7 Top-K (fail-fast)
Bắt buộc: cùng test users, đúng shape `(n_test_users,20)`, đúng item IDs, không trùng item trong một list, không chứa Train items, điểm hợp lệ. Kiểm tra `export_verification.json` nếu có (Most Popular có thể không có).

**Không tự tin tưởng metadata:** riêng train-mask được kiểm tra thực tế trực tiếp trên Train CSR. Nếu phát hiện bất nhất sẽ dừng ngay thay vì âm thầm tạo E2 metrics.

In [ ]:
def topk_binary_hits(interactions_csr, users, ranked_items):
    # Sparse 2-D pairwise indexing; 9860 x 20 is small enough to materialize.
    hits = interactions_csr[users[:, None], ranked_items]
    return (hits.toarray() if sp.issparse(hits) else np.asarray(hits)).astype(bool)

export_data = {}
audit_rows = []

for model, runs in EXPECTED_EXPORTS.items():
    for seed, path in runs.items():
        with np.load(path, allow_pickle=False) as data:
            required = {'user_ids','item_ids','scores'}
            assert required.issubset(set(data.files)), f'{path}: missing {required-set(data.files)}'
            users = data['user_ids'].astype(np.int64, copy=True)
            items = data['item_ids'].astype(np.int64, copy=True)
            scores = np.asarray(data['scores']).copy()

        assert users.ndim == 1 and items.ndim == 2 and scores.ndim == 2, path
        assert items.shape == scores.shape == (len(users), 20), f'{path}: wrong shape'
        assert len(np.unique(users)) == len(users), f'{path}: duplicate users'
        assert np.array_equal(np.sort(users), active_test_users), f'{path}: test users differ from dataset'
        assert bool(np.all((items >= 0) & (items < n_items))), f'{path}: invalid item IDs'
        assert not np.any(np.diff(np.sort(items, axis=1), axis=1) == 0), f'{path}: duplicate recommendations'
        assert np.all(np.isfinite(scores)), f'{path}: non-finite scores'
        assert not topk_binary_hits(train, users, items).any(), f'{path}: Train item leaked into Top-K!'

        # Normalize user order, preserving each user's ranked Top-K.
        perm = np.argsort(users)
        users, items, scores = users[perm], items[perm], scores[perm]
        meta_path = path.parent / 'export_meta.json'
        verify_path = path.parent / 'export_verification.json'
        meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
        verification = json.loads(verify_path.read_text()) if verify_path.exists() else {}
        if 'total_items' in meta:
            assert int(meta['total_items']) == n_items, f'{path}: catalog size discrepancy'
        if 'top_k' in meta:
            assert int(meta['top_k']) == 20, f'{path}: unexpected K'
        if 'masked_splits' in meta:
            assert meta['masked_splits'] == ['train'], f'{path}: unexpected masking protocol'
        if 'test_labels_used_to_rank' in meta:
            assert meta['test_labels_used_to_rank'] is False, f'{path}: metadata flags test label leakage'
        if verification:
            assert verification.get('status') == 'passed', f'{path}: failed E1 verification'
            for check_name, comparison in verification.get('compared_metrics', {}).items():
                assert comparison.get('passed') is True, f'{path}: E1 mismatch: {check_name}'

        export_data[(model, seed)] = {'users': users, 'items': items, 'scores': scores,
                                      'meta': meta, 'verification': verification}
        audit_rows.append({
            'model': model, 'seed': seed, 'n_users': len(users),
            'top_k': items.shape[1], 'unique_users': True,
            'same_test_users': True, 'no_train_items': True, 'unique_items_per_user': True,
            'metadata_masked_splits': ','.join(meta.get('masked_splits', [])) if meta else '',
            'verification_status': verification.get('status', 'not available'),
            'npz_file': str(path),
        })

audit = pd.DataFrame(audit_rows)
audit.to_csv(OUT_DIR/'E2_topk_audit.csv', index=False)
print('AUDIT PASSED:', len(audit),'Top-K files; all have identical test users and Train-only masking.')
display(audit[['model','seed','n_users','top_k','no_train_items','verification_status']])

## CELL 5 — E1 consistency gate (Recall/NDCG từ chính Top-K)
Trước khi phân tích bias, tính lại **standard macro Recall/NDCG** để kiểm tra kết quả tái hiện có khớp E1. Kiểm tra với `export_verification.json` (nếu có) và `e1_baseline_summary.json` ở E1 (nếu có). Sai khác quá `1e-5` sẽ dừng.

Nếu logic metric của source gốc khác chuẩn công thức dưới đây, notebook sẽ báo sai khác để chúng ta kiểm tra **chứ không tự chỉnh** E1.

In [ ]:
def compute_standard_accuracy(users, items, test_csr, k):
    rank_hits = topk_binary_hits(test_csr, users, items[:, :k])
    n_gt = np.diff(test_csr.indptr)[users]
    assert np.all(n_gt > 0)
    recall = np.mean(rank_hits.sum(axis=1) / n_gt)
    discounts = 1.0 / np.log2(np.arange(2, k+2))
    dcg = (rank_hits * discounts).sum(axis=1)
    ideal = np.r_[0.0, np.cumsum(discounts)]
    idcg = ideal[np.minimum(n_gt, k)]
    ndcg = np.mean(dcg / idcg)
    return float(recall), float(ndcg)

# Optional raw run metrics from original E1 summary (if available).
e1_expected = {}
if E1_SUMMARY_PATH.exists():
    e1_raw = json.loads(E1_SUMMARY_PATH.read_text()).get('raw_runs', [])
    for row in e1_raw:
        model = str(row['Method'])
        seed = str(row.get('Seed', '-'))
        e1_expected[(model, seed)] = row
    print('Loaded E1 summary:', E1_SUMMARY_PATH)
else:
    print('Note: E1 summary JSON missing; verification JSONs will be checked where present.')

accuracy_rows = []
TOLERANCE = 1e-5
for (model, seed), exp in export_data.items():
    users, items = exp['users'], exp['items']
    for k in K_VALUES:
        r, n = compute_standard_accuracy(users, items, test, k)
        row = {'model':model, 'seed':seed, 'K':k, 'Recall':r, 'NDCG':n}
        for metric, actual in [('Recall', r),('NDCG', n)]:
            key = f'{metric.lower()}@{k}'
            verified_value = exp['verification'].get('test_metrics_from_export', {}).get(key)
            if verified_value is not None:
                assert abs(actual-float(verified_value)) <= TOLERANCE, (
                    f'{model}, seed {seed}, {key}: actual {actual}, verification {verified_value}'
                )
            expected_row = e1_expected.get((model,seed))
            if expected_row is not None:
                expected_value = expected_row.get(f'{metric}@{k}')
                if expected_value is not None:
                    assert abs(actual-float(expected_value)) <= TOLERANCE, (
                        f'{model}, seed {seed}, {key}: actual {actual}, E1 {expected_value}'
                    )
        accuracy_rows.append(row)
accuracy_audit = pd.DataFrame(accuracy_rows)
accuracy_audit.to_csv(OUT_DIR/'E2_accuracy_recheck.csv', index=False)
print('Accuracy consistency checks PASSED (for available E1 references).')
display(accuracy_audit)

## CELL 6 — Popularity metrics E2

- `ARP@K = mean_u [mean_{i in Rec_u@K} degree_train(i)]`
- `APLT@K = mean_u [#(Rec_u@K ∩ Tail) / K]`
- `HeadRecall@K = mean_{u: |GT_u∩Head|>0} [#(Rec_u@K∩GT_u∩Head) / |GT_u∩Head|]`
- `TailRecall@K`: tương tự cho Tail
- `Spearman(degree, exposure)`: tính trên **toàn catalog** gồm cả item được gợi ý 0 lần, có giá trị tham khảo cho sự lệ thuộc vào popularity.

Đơn vị phân tích head/tail recall là **user đủ điều kiện**, không gán 0 cho user không có ground truth thuộc nhóm đó.

In [ ]:
def conditional_macro_recall(hits, groundtruth_counts):
    eligible = groundtruth_counts > 0
    if not eligible.any():
        return np.nan, 0
    rates = hits[eligible] / groundtruth_counts[eligible]
    return float(np.mean(rates)), int(eligible.sum())

metrics_rows = []
exposure_rows = []
for (model, seed), exp in export_data.items():
    users, ranked = exp['users'], exp['items']
    gt = test[users]
    head_gt_count = np.asarray(gt[:, head_mask].sum(axis=1)).ravel()
    tail_gt_count = np.asarray(gt[:, tail_mask].sum(axis=1)).ravel()

    for k in K_VALUES:
        items = ranked[:, :k]
        is_hit = topk_binary_hits(test, users, items)
        head_hits = (is_hit & head_mask[items]).sum(axis=1)
        tail_hits = (is_hit & tail_mask[items]).sum(axis=1)
        head_recall, head_eligible = conditional_macro_recall(head_hits, head_gt_count)
        tail_recall, tail_eligible = conditional_macro_recall(tail_hits, tail_gt_count)

        exposure = np.bincount(items.ravel(), minlength=n_items).astype(np.int64)
        assert exposure.sum() == len(users)*k
        rho, rho_p = spearmanr(item_degree, exposure)
        # Pearson on log-degrees/exposure (optional descriptive statistic)
        lx, ly = np.log1p(item_degree), np.log1p(exposure)
        log_pearson = float(np.corrcoef(lx,ly)[0,1]) if lx.std()>0 and ly.std()>0 else np.nan

        metrics_rows.append({
            'model':model, 'seed':seed, 'K':k, 'n_test_users':len(users),
            'ARP': float(item_degree[items].mean()),
            'APLT': float(tail_mask[items].mean()),
            'HeadRecall': head_recall,
            'TailRecall': tail_recall,
            'n_head_eligible_users': head_eligible,
            'n_tail_eligible_users': tail_eligible,
            'HeadExposureShare': float(exposure[head_mask].sum()/exposure.sum()),
            'Coverage': float((exposure > 0).mean()),
            'DegreeExposureSpearman': float(rho),
            'DegreeExposureSpearmanP': float(rho_p),
            'DegreeExposureLogPearson': log_pearson,
        })
        exposure_rows.append(pd.DataFrame({
            'model':model, 'seed':seed, 'K':k, 'item_id':np.arange(n_items),
            'train_degree':item_degree, 'group':np.where(head_mask, 'Head', 'Tail'),
            'exposure':exposure,
        }))

seed_metrics = pd.DataFrame(metrics_rows)
seed_metrics.to_csv(OUT_DIR/'E2_seed_metrics.csv', index=False)
exposure_metrics = pd.concat(exposure_rows, ignore_index=True)
exposure_metrics.to_csv(OUT_DIR/'E2_item_exposure_per_run.csv', index=False)
print('Per-run E2 metrics saved:', len(seed_metrics),'rows')
display(seed_metrics[['model','seed','K','ARP','APLT','HeadRecall','TailRecall',
                      'n_head_eligible_users','n_tail_eligible_users','DegreeExposureSpearman']].round(6))

## CELL 7 — Tổng hợp mean ± SD theo seeds (`ddof=0`)
Most Popular có **1 run, SD = NaN**, không diễn giải là độ biến thiên bằng 0. Với 3 seeds của LightGCN / RLMRec-Con, `SD` là độ lệch chuẩn của kết quả giữa các seeds.

In [ ]:
metric_cols = ['ARP','APLT','HeadRecall','TailRecall','HeadExposureShare',
               'Coverage','DegreeExposureSpearman','DegreeExposureLogPearson']

summary_rows = []
for (model,k), part in seed_metrics.groupby(['model','K'], sort=False):
    row = {'model':model,'K':int(k),'n_seeds':len(part),
           'n_test_users':int(part['n_test_users'].iloc[0]),
           'n_head_eligible_users':int(part['n_head_eligible_users'].iloc[0]),
           'n_tail_eligible_users':int(part['n_tail_eligible_users'].iloc[0])}
    for m in metric_cols:
        a = part[m].to_numpy(dtype=float)
        row[m+'_mean'] = float(np.mean(a))
        row[m+'_sd'] = float(np.std(a,ddof=0)) if len(a)>1 else np.nan
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary['model'] = pd.Categorical(summary['model'], MODELS, ordered=True)
summary = summary.sort_values(['model','K']).reset_index(drop=True)
summary['model'] = summary['model'].astype(str)
summary.to_csv(OUT_DIR/'E2_model_summary.csv', index=False)

format_view = summary[['model','K','n_seeds','n_head_eligible_users','n_tail_eligible_users']].copy()
for m in ['ARP','APLT','HeadRecall','TailRecall','DegreeExposureSpearman']:
    def display_mean_sd(row):
        value = row[m+'_mean']
        sd = row[m+'_sd']
        return f'{value:.6f} ± {sd:.6f}' if row['n_seeds'] > 1 else f'{value:.6f}'
    format_view[m] = summary.apply(display_mean_sd, axis=1)
format_view.to_csv(OUT_DIR/'E2_model_summary_formatted.csv', index=False)
print('E2 summary (mean ± population SD across seeds):')
display(format_view)

# Item-level exposure mean across seeds, useful for trend plots/reproducibility.
item_exposure_summary = (exposure_metrics
    .groupby(['model','K','item_id','train_degree','group'], sort=False)['exposure']
    .agg(exposure_mean='mean', exposure_sd=lambda x: float(np.std(x,ddof=0)) if len(x)>1 else np.nan)
    .reset_index())
item_exposure_summary.to_csv(OUT_DIR/'E2_item_exposure_summary.csv',index=False)

## CELL 8 — Figure 1: ARP & APLT (@10/@20)
Đồ thị mô tả mức độ popularity bias ở danh sách recommendation; error bars thể hiện **SD giữa các seeds**, không phải confidence interval.

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))
x = np.arange(len(MODELS))
width = 0.34
for ax, (metric,ylabel) in zip(axes,[('ARP','Train interaction count'),('APLT','Fraction of Tail items')]):
    for shift,k in [(-width/2,10),(width/2,20)]:
        rows = summary[summary['K']==k].set_index('model').loc[MODELS]
        means = rows[metric+'_mean'].to_numpy(dtype=float)
        sds = rows[metric+'_sd'].fillna(0).to_numpy(dtype=float)
        ax.bar(x+shift, means, width, yerr=sds, capsize=4, label=f'@{k}')
    ax.set_xticks(x, MODELS, rotation=10)
    ax.set_ylabel(ylabel)
    ax.set_title(metric + '@K')
    ax.grid(axis='y',alpha=0.25,linestyle='--')
    ax.set_axisbelow(True)
    ax.legend()
    if metric=='APLT': ax.set_ylim(0,1)
fig.suptitle('E2 — Recommendation Popularity Bias', fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR/'E2_ARP_APLT.png',dpi=300,bbox_inches='tight')
fig.savefig(OUT_DIR/'E2_ARP_APLT.pdf',bbox_inches='tight')
plt.show()

## CELL 9 — Figure 2: Conditional Head / Tail Recall (@20)
Cách nhìn này phân biệt **sự cải thiện trên item phổ biến** và **sự cải thiện trên item ít phổ biến**. Hai nhóm users đủ điều kiện có thể khác nhau; xem cột `n_head_eligible_users` và `n_tail_eligible_users`.

In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
rows = summary[summary['K']==20].set_index('model').loc[MODELS]
x = np.arange(len(MODELS)); width=0.36
for shift,metric,label in [(-width/2,'HeadRecall','Head Recall@20'),
                           (width/2,'TailRecall','Tail Recall@20')]:
    ax.bar(x+shift, rows[metric+'_mean'], width,
           yerr=rows[metric+'_sd'].fillna(0), capsize=4, label=label)
ax.set_xticks(x, MODELS)
ax.set_ylabel('Conditional macro Recall@20')
ax.set_title('E2 — Accuracy by Item Popularity Group')
ax.grid(axis='y',alpha=0.25,linestyle='--')
ax.set_axisbelow(True); ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR/'E2_Head_Tail_Recall20.png',dpi=300,bbox_inches='tight')
fig.savefig(OUT_DIR/'E2_Head_Tail_Recall20.pdf',bbox_inches='tight')
plt.show()

## CELL 10 — Figure 3: Degree–Exposure (@20)
Mỗi điểm là một item. Trục log(1+x) giúp nhìn cả item degree thấp / exposure thấp. Với hai mô hình 3 seeds, mỗi item được biểu diễn bằng exposure **trung bình qua seeds**. Không nên diễn giải tương quan này là nhân quả.

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,4.5),sharex=True,sharey=True)
for ax,model in zip(axes,MODELS):
    part = item_exposure_summary[
        (item_exposure_summary['model']==model) & (item_exposure_summary['K']==20)
    ].sort_values('item_id')
    xlog = np.log1p(part['train_degree'].to_numpy())
    ylog = np.log1p(part['exposure_mean'].to_numpy())
    ax.scatter(xlog,ylog,alpha=0.12,s=5,rasterized=True)
    rho = float(summary[(summary.model==model)&(summary.K==20)]
                ['DegreeExposureSpearman_mean'].iloc[0])
    ax.set_title(f'{model}\nSpearman ρ = {rho:.3f}')
    ax.set_xlabel('log(1 + train degree)')
    ax.grid(alpha=0.2,linestyle='--')
axes[0].set_ylabel('log(1 + mean exposure@20)')
fig.suptitle('E2 — Item Popularity vs Recommendation Exposure',fontsize=14)
fig.tight_layout()
fig.savefig(OUT_DIR/'E2_Degree_Exposure20.png',dpi=300,bbox_inches='tight')
fig.savefig(OUT_DIR/'E2_Degree_Exposure20.pdf',bbox_inches='tight')
plt.show()

## CELL 11 — Accuracy–Popularity Trade-off (RQ1) & relative changes
Đối chiếu Recall@20 của E1 với APLT@20 và ARP@20 của E2. **Mọi kết quả E2 là dữ liệu mới từ Top-K đã lưu**, còn Recall/NDCG được tính lại ở CELL 5 để bảo đảm cùng test users / labels. Không kết luận giảm bias chỉ từ một chỉ số.

In [ ]:
accuracy_summary = (
    accuracy_audit.groupby(['model','K'],as_index=False)[['Recall','NDCG']].mean()
)
tradeoff = summary.merge(accuracy_summary,on=['model','K'],validate='one_to_one')
tradeoff.to_csv(OUT_DIR/'E2_accuracy_popularity_tradeoff.csv',index=False)

fig,ax = plt.subplots(figsize=(8,5.3))
part = tradeoff[tradeoff.K==20].set_index('model').loc[MODELS]
for model,row in part.iterrows():
    ax.scatter(row['APLT_mean'],row['Recall'],s=85)
    ax.annotate(model,(row['APLT_mean'],row['Recall']),
                xytext=(6,5),textcoords='offset points')
ax.set_xlabel('APLT@20 (higher = more Tail recommendations)')
ax.set_ylabel('Recall@20 (higher = better accuracy)')
ax.set_title('E2 — Accuracy–Popularity Trade-off')
ax.grid(alpha=0.25,linestyle='--')
fig.tight_layout()
fig.savefig(OUT_DIR/'E2_Accuracy_APLT_Tradeoff20.png',dpi=300,bbox_inches='tight')
fig.savefig(OUT_DIR/'E2_Accuracy_APLT_Tradeoff20.pdf',bbox_inches='tight')
plt.show()

base = part.loc['LightGCN']
new = part.loc['RLMRec-Con']
relative_rows=[]
for metric,desired in [('Recall','higher'),('NDCG','higher'),('ARP_mean','lower (descriptive)'),
                       ('APLT_mean','higher'),('TailRecall_mean','higher')]:
    old, current = float(base[metric]),float(new[metric])
    relative_rows.append({'metric':metric,'direction':desired,
        'LightGCN':old,'RLMRec-Con':current,
        'absolute_change':current-old,
        'relative_change_pct':100*(current-old)/old if old!=0 else np.nan})
comparison = pd.DataFrame(relative_rows)
comparison.to_csv(OUT_DIR/'E2_RLMRec_vs_LightGCN_20.csv',index=False)
print('RQ1 comparison @20 (positive change is not always desirable for every metric):')
display(comparison.round(5))

## CELL 12 — Ghi lại protocol & danh sách output
Notebook tạo `E2_protocol.json` / `E2_protocol.md` trong Drive để cả nhóm thống nhất metric và head/tail split. **Không có số liệu giả định** trong các output; chỉ ghi kết quả chạy thực tế.

In [ ]:
protocol = {
    'project':'SNA-MXH', 'experiment':'E2_Popularity_Bias',
    'dataset':'amazon', 'root':str(ROOT),
    'evaluation':'full ranking; existing Top-20 test exports from E1',
    'ranking_mask':'train_only', 'validation_masked':False,
    'test_labels_used_for_ranking':False,  # documented from exporter metadata, not reranked here
    'popularity_source':'binary trn_mat.pkl only',
    'head_fraction':HEAD_FRACTION,
    'head_count_rule':'ceil(head_fraction * n_items)',
    'sort_tie_rule':'item ID ascending',
    'head_item_count':int(n_head),'tail_item_count':int(n_items-n_head),
    'num_users':int(n_users),'num_items':int(n_items),
    'num_test_users':int(len(active_test_users)),
    'k_values':list(K_VALUES),
    'head_tail_recall':'conditional macro over users with >=1 group-specific Test positive',
    'arp':'average train degree in Top-K, macro user average',
    'aplt':'average share of Tail recommendations in Top-K, macro user average',
    'degree_exposure':'Spearman over all catalog items, including zero exposure',
    'seeds':[2023,2024,2025], 'aggregate':'mean + population SD (ddof=0)',
    'most_popular':'single run; no SD',
    'e1_check_tolerance':TOLERANCE,
    'models':MODELS,
}
(OUT_DIR/'E2_protocol.json').write_text(
    json.dumps(protocol,indent=2,ensure_ascii=False),encoding='utf-8')
(OUT_DIR/'E2_protocol.md').write_text(
    '# E2 — Popularity Bias Evaluation Protocol\n\n'
    'Train-only masking; no additional validation masking. '
    'Train interactions (binary) define item degree and a top-20% Head set '
    '(ceil, tie-break by item ID). Tail is the other 80%. '
    'Metrics: ARP@10/20, APLT@10/20, conditional macro Head/Tail Recall@10/20, '
    'degree–exposure Spearman; run-level and mean ± SD (ddof=0). '
    'Most Popular has one run and no SD. '
    'All metrics use the existing 7 Top-K exports; no retraining.\n',
    encoding='utf-8')

print('DONE. Files in',OUT_DIR)
for p in sorted(OUT_DIR.glob('E2_*')):
    print(' -',p.name)
print('\nNext: send E2_model_summary_formatted.csv and E2_RLMRec_vs_LightGCN_20.csv for interpretation.')
